# FASTA, FASTQ e SRA — dados brutos de *Hypochilus*

Dataset modelo: Ciaccio, Debray & Hedin (2022)  
BioProject: **PRJNA760946**  
Run: **SRR15736591** — *Hypochilus petrunkevitchi*

O run completo possui cerca de 1,76 milhão de spots. Para aula, baixaremos uma fração.

## 1. Montar o Drive e criar a pasta

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
BASE = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular/03_sra_fastq")
BASE.mkdir(parents=True, exist_ok=True)
print(BASE)

## 2. Parâmetros didáticos

In [ ]:
SRR = "SRR15736591"
MAX_SPOTS = 200_000

print("Run:", SRR)
print("Máximo de spots:", f"{MAX_SPOTS:,}")

## 3. Instalar SRA Toolkit

In [ ]:
!apt-get -qq update
!apt-get -qq install -y sra-toolkit
!fasterq-dump --version

## 4. Baixar uma fração paired-end

`-X` limita o maior spot utilizado.  
`--split-files` separa as duas leituras em `_1.fastq` e `_2.fastq`.

In [ ]:
outdir = str(BASE)
!fasterq-dump "$SRR" -X "$MAX_SPOTS" --split-files -e 2 -O "$outdir"

## 5. Compactar os FASTQ

In [ ]:
!gzip -f "{BASE}/{SRR}_1.fastq"
!gzip -f "{BASE}/{SRR}_2.fastq"
!ls -lh "{BASE}"

## 6. Visualizar dois registros FASTQ

In [ ]:
!zcat "{BASE}/{SRR}_1.fastq.gz" | head -8

## 7. Contar reads em R1 e R2

In [ ]:
import gzip

def contar_reads(path):
    n = 0
    with gzip.open(path, "rt") as f:
        for _ in f:
            n += 1
    return n // 4

r1 = BASE / f"{SRR}_1.fastq.gz"
r2 = BASE / f"{SRR}_2.fastq.gz"

print("R1:", contar_reads(r1))
print("R2:", contar_reads(r2))

## 8. Conferir o pareamento

Os identificadores de R1 e R2 devem corresponder aos mesmos spots.

In [ ]:
def primeiro_id(path):
    with gzip.open(path, "rt") as f:
        return f.readline().strip()

print("R1:", primeiro_id(r1))
print("R2:", primeiro_id(r2))

## Resultado esperado

A pasta `03_sra_fastq` deve conter:

- `SRR15736591_1.fastq.gz`
- `SRR15736591_2.fastq.gz`

Esses arquivos serão reutilizados na aula de FastQC e trimming.